<a id='lambda'></a>

## 9. 🧩 Pattern 9: lambda — Inline Anonymous Functions — LC 56, 57, 179, 215, 347, 692, 973, 1235

---

```
PROBLEM:
  LC 56   — Merge Intervals: sort by start → sorted(intervals, key=lambda x: x[0])
  LC 57   — Insert Interval: sort before inserting
  LC 179  — Largest Number: custom comparator — key=lambda x: str(x) * 3
  LC 215  — Kth Largest Element: sort with key for custom ordering
  LC 347  — Top K Frequent: sort by frequency → sorted(d.items(), key=lambda x: -x[1])
  LC 692  — Top K Frequent Words: sort by (-freq, word)
  LC 973  — K Closest Points: sort by distance² → key=lambda p: p[0]**2 + p[1]**2
  LC 1235 — Max Profit Job Schedule: sort jobs by end time

SYNTAX:
  lambda args: expr

  Regular def:                    Lambda equivalent:
  def square(x): return x*x       lambda x: x*x
  def add(a, b): return a+b       lambda a, b: a+b
  def first(t): return t[0]       lambda t: t[0]

SINGLE EXPRESSION ONLY — lambda cannot contain:
  ❌  lambda x: if x > 0: return x   (no if-blocks)
  ❌  lambda x: y = x*2; return y    (no assignments)
  ❌  lambda x: for i in x: ...      (no loops)
  ✅  lambda x: x if x > 0 else 0    (ternary expression — one line, one value)

USED AS key= ARGUMENT — the most common interview use:
  sorted(nums, key=lambda x: -x)               ← sort descending
  sorted(words, key=lambda w: len(w))          ← sort by length
  sorted(intervals, key=lambda x: x[0])        ← sort by first element
  min(points, key=lambda p: p[0]**2+p[1]**2)  ← closest to origin
  max(items, key=lambda x: x[1])               ← max by second element

MULTI-KEY SORT — tuple as key:
  sorted(words, key=lambda w: (-freq[w], w))   ← primary: freq desc, secondary: alpha
  sorted(jobs, key=lambda j: (j[1], j[0]))     ← primary: end time, secondary: start

LAMBDA vs DEF — when to use which:
  Use lambda when:
    - used exactly once (inline, as argument)
    - body is a single simple expression
    - no name needed
  Use def when:
    - called in more than one place
    - body needs multiple steps or statements
    - needs a docstring or meaningful name

SLOW MOTION TRACE — sorted(intervals, key=lambda x: x[0]):
  intervals = [[3,6],[1,3],[2,4]]
  keys extracted: [3, 1, 2]
  sorted by keys: [[1,3],[2,4],[3,6]]
  lambda called once per element — never stores result, just directs comparison

KEY INSIGHT:
  lambda is a throwaway key extractor. It tells sorted/min/max:
  "don't compare the objects directly — compare this derived value instead."

TIME / SPACE:
  Time:  O(n log n) for sort — lambda itself is O(1) per call
  Space: O(n) — sorted() creates a new list
```

In [ ]:
# Pattern 9: lambda
# Throwaway key extractor — tells sorted/min/max what value to compare.

# 1. basic lambda forms
square   = lambda x: x * x
add      = lambda a, b: a + b
clamp    = lambda x: x if x > 0 else 0          # ternary — only valid multi-branch form
print(f"square(5)    : {square(5)}")
print(f"add(3,4)     : {add(3, 4)}")
print(f"clamp(-3)    : {clamp(-3)}")
print(f"clamp(7)     : {clamp(7)}")

# 2. sorted() with key= — the primary interview use
intervals = [[3, 6], [1, 3], [2, 4]]
by_start  = sorted(intervals, key=lambda x: x[0])    # sort by first element
by_end    = sorted(intervals, key=lambda x: x[1])    # sort by second element
desc      = sorted([5, 2, 8, 1], key=lambda x: -x)  # sort descending
print(f"by start  : {by_start}")
print(f"by end    : {by_end}")
print(f"descending: {desc}")

# 3. multi-key sort — tuple as key (primary, secondary)
from collections import Counter
words = ["banana", "apple", "fig", "cherry", "ant", "fig", "banana", "banana"]
freq  = Counter(words)
# sort by: frequency desc (-freq), then alphabetically asc (word)
ranked = sorted(set(words), key=lambda w: (-freq[w], w))
print(f"freq      : {dict(freq)}")
print(f"ranked    : {ranked}")

# 4. min/max with key=
points = [[3, 3], [5, -1], [-2, 4], [1, 1]]
closest = min(points, key=lambda p: p[0]**2 + p[1]**2)  # closest to origin
print(f"closest to origin: {closest}")

# 5. drill — sort list of tuples by second element, then first
data = [(3, 2), (1, 5), (2, 2), (1, 3), (2, 1)]
result = sorted(data, key=lambda t: (t[1], t[0]))        # secondary: t[0], primary: t[1]
# slow motion on key extraction:
# (3,2) → (2,3)
# (1,5) → (5,1)
# (2,2) → (2,2)
# (1,3) → (3,1)
# (2,1) → (1,2)
# sorted: (1,2) (2,2) (2,3) (3,1) (5,1) → [(2,1),(2,2),(3,2),(1,3),(1,5)]
print(f"sorted by (t[1],t[0]): {result}")


def k_closest_points(points: list, k: int) -> list:
    """
    LC 973 — K Closest Points to Origin
    Approach: sort by distance squared (no sqrt needed — monotone transform).
    Args:
        points (list[list[int]]): list of [x, y] coordinates.
        k (int): number of closest points to return.
    Returns:
        list[list[int]]: k points nearest to origin.
    Time:  O(n log n) — full sort; O(n log k) with heap
    Space: O(n) — sorted() allocates new list
    """
    return sorted(points, key=lambda p: p[0]**2 + p[1]**2)[:k]

    # slow motion on points=[[3,3],[5,-1],[-2,4]], k=2:
    # keys: 3²+3²=18,  5²+1²=26,  2²+4²=20
    # sorted: [[3,3](18), [-2,4](20), [5,-1](26)]
    # [:2] → [[3,3], [-2,4]]


def merge_intervals(intervals: list) -> list:
    """
    LC 56 — Merge Intervals
    Approach: sort by start time, then greedily merge overlapping intervals.
    Args:
        intervals (list[list[int]]): list of [start, end] intervals.
    Returns:
        list[list[int]]: merged non-overlapping intervals.
    Time:  O(n log n) — dominated by sort
    Space: O(n) — output list
    """
    intervals.sort(key=lambda x: x[0])   # lambda sorts by start — the whole trick
    merged = [intervals[0]]

    # slow motion on [[1,3],[2,6],[8,10],[15,18]]:
    # start=merged=[[1,3]]
    # [2,6]: 2<=3 overlap → extend end to max(3,6)=6  → [[1,6]]
    # [8,10]: 8>6 no overlap → append              → [[1,6],[8,10]]
    # [15,18]: 15>10 no overlap → append            → [[1,6],[8,10],[15,18]]

    for start, end in intervals[1:]:
        if start <= merged[-1][1]:              # overlaps with last merged interval
            merged[-1][1] = max(merged[-1][1], end)   # extend the end
        else:
            merged.append([start, end])         # no overlap — new interval
    return merged


def test_harness(fn):
    tests = [
        ([[1, 3], [2, 6], [8, 10], [15, 18]], [[1, 6], [8, 10], [15, 18]]),
        ([[1, 4], [4, 5]],                     [[1, 5]]),
        ([[1, 4], [2, 3]],                     [[1, 4]]),
        ([[1, 4]],                              [[1, 4]]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")


test_harness(merge_intervals)

print(k_closest_points([[3, 3], [5, -1], [-2, 4]], 2))   # [[3,3],[-2,4]]
print(k_closest_points([[1, 3], [-2, 2]], 1))             # [[-2,2]]

print("lambda defined.")